# Layer 1 — Adversarial Agent Fine-Tuning DziriBERT

## 1. Setup & Install

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn statsmodels
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%matplotlib inline
import os, re, json, random, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                              confusion_matrix, classification_report, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight
from statsmodels.stats.proportion import proportion_confint
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           Trainer, TrainingArguments, EarlyStoppingCallback)

sns.set_theme(style="whitegrid")
print("Imports OK. CUDA available:", torch.cuda.is_available())


## 2. Config

In [ ]:
SEED = 42
SEEDS = [42, 43, 44]
MAX_LENGTH = 128
MODEL_CHECKPOINT = "alger-ia/dziribert"  

DATASETS_BASE = "/content/drive/MyDrive/Datasets"
LAYER1_DIR    = f"{DATASETS_BASE}/layer1"
OSACT_DIR     = f"{DATASETS_BASE}/OSACT"

LAYER1_TRAIN_PATH = f"{LAYER1_DIR}/layer1_train.csv"
LAYER1_VAL_PATH   = f"{LAYER1_DIR}/layer1_validation.csv"
LAYER1_TEST_PATH  = f"{LAYER1_DIR}/layer1_test.csv"
DEV_OSACT_PATH    = f"{OSACT_DIR}/dev_osact.csv"   
OUT_DIR    = f"{LAYER1_DIR}/outputs/dziribert_adversarial_rebuild"
MODEL_OUT  = f"{OUT_DIR}/model"        
REPORT_DIR = f"{OUT_DIR}/reports"
for d in [OUT_DIR, MODEL_OUT, REPORT_DIR]:
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 3. Load Data & Derive `prompt_injection`-Only Labels

In [ ]:
ID2LABEL = {0: "safe", 1: "prompt_injection"}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_LABELS = 2

def load_adversarial_split(path, split_name):
    df = pd.read_csv(path)
    before = len(df)
    df = df[df["original_label"].notna() & (df["original_label"] != "tool_misuse")].copy()
    excluded = before - len(df)
    # kept as-is when original_label == "prompt_injection" (no rename); everything else -> "safe"
    df["adv_label"] = df["original_label"].apply(lambda x: "prompt_injection" if x == "prompt_injection" else "safe")
    df["label_id"] = df["adv_label"].map(LABEL2ID).astype(int)
    print(f"[{split_name}] kept {len(df)} / {before} rows ({excluded} excluded: tool_misuse or blind-NaN)")
    print(f"  class counts: {df['adv_label'].value_counts().to_dict()}")
    return df

train_df = load_adversarial_split(LAYER1_TRAIN_PATH, "train")
val_df   = load_adversarial_split(LAYER1_VAL_PATH,   "validation")
test_df  = load_adversarial_split(LAYER1_TEST_PATH,  "test")

text_col = "text_clean" if "text_clean" in train_df.columns else "text"
print("Using text column:", text_col)

def fill_text_fallback(df, primary_col, fallback_col="text"):
    """Ported unchanged from the Intent Agent notebook. AAAC rows never populate
    text_clean (verified: 0/2472 AAAC train rows contain any OSACT-style noise
    marker — @USER/<LF>/RT/URL — so the column was simply never filled for a
    source that never needed cleaning). Since prompt_injection (this agent's
    ENTIRE positive class) is 100% AAAC-origin, skipping this fallback would
    tokenize every prompt_injection-labeled row as an empty string."""
    if primary_col == fallback_col or fallback_col not in df.columns:
        return df
    is_bad = df[primary_col].isna() | (df[primary_col].astype(str).str.strip() == "")
    n_filled = 0
    if is_bad.any():
        can_fill = is_bad & df[fallback_col].notna() & (df[fallback_col].astype(str).str.strip() != "")
        df.loc[can_fill, primary_col] = df.loc[can_fill, fallback_col]
        n_filled = int(can_fill.sum())
    still_bad = df[primary_col].isna() | (df[primary_col].astype(str).str.strip() == "")
    print(f"  filled {n_filled} rows from '{fallback_col}' (AAAC rows, already clean), "
          f"{int(still_bad.sum())} still empty (dropped)")
    return df[~still_bad].copy()

print("Applying text_clean fallback (CRITICAL here — prompt_injection is 100% AAAC-origin):")
train_df = fill_text_fallback(train_df, text_col)
val_df   = fill_text_fallback(val_df, text_col)
test_df  = fill_text_fallback(test_df, text_col)

dev_osact_adv = pd.read_csv(DEV_OSACT_PATH)
dev_osact_adv = dev_osact_adv[dev_osact_adv["layer1_label"].notna()].copy()
dev_osact_adv["label_id"] = LABEL2ID["safe"]
if text_col in dev_osact_adv.columns:
    dev_osact_adv = fill_text_fallback(dev_osact_adv, text_col)
print(f"[dev_osact, relabeled safe-only for OOD pool] {len(dev_osact_adv)} rows")


## 4. Class Weights

In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.arange(NUM_LABELS), y=train_df["label_id"].values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights (safe/prompt_injection):", class_weights.round(3))


## 5. Dataset, Weighted Trainer, Metrics (identical pattern to Intent Agent notebook)

In [6]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.encodings = tokenizer(list(texts), truncation=True, padding="max_length", max_length=max_length)
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


class WeightedLossTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = class_weights_tensor
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights_tensor.to(logits.device) if self.class_weights_tensor is not None else None
        loss = torch.nn.CrossEntropyLoss(weight=weight)(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    _, _, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": macro_f1}


## 6. Multi-Seed Training

In [ ]:
def train_adv_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    print(f"\n{'='*70}\n Training Adversarial Agent — seed={seed}\n{'='*70}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_CHECKPOINT, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE)

    train_ds = TextClassificationDataset(train_df[text_col], train_df["label_id"], tokenizer)
    val_ds   = TextClassificationDataset(val_df[text_col],   val_df["label_id"],   tokenizer)

    args = TrainingArguments(
        output_dir=f"{OUT_DIR}/tmp_seed{seed}",
        num_train_epochs=6,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_steps=50,
        seed=seed,
        report_to="none",
        fp16=torch.cuda.is_available(),     
        dataloader_num_workers=2,           
    )

    trainer = WeightedLossTrainer(
        model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
        compute_metrics=compute_metrics, class_weights_tensor=class_weights_tensor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0
    val_metrics = trainer.evaluate()
    return trainer, tokenizer, val_metrics, train_time


seed_results = {}
for seed in SEEDS:
    trainer, tokenizer, val_metrics, train_time = train_adv_seed(seed)
    seed_results[seed] = {"trainer": trainer, "tokenizer": tokenizer,
                          "val_macro_f1": val_metrics["eval_macro_f1"], "train_time_s": train_time}
    print(f"[seed={seed}] val_macro_f1={val_metrics['eval_macro_f1']:.4f}  train_time={train_time:.0f}s")

best_seed = max(seed_results, key=lambda s: seed_results[s]["val_macro_f1"])
f1_values = [seed_results[s]["val_macro_f1"] for s in SEEDS]
print(f"\nBest seed: {best_seed}  (val_macro_f1={seed_results[best_seed]['val_macro_f1']:.4f})")
print(f"Seed variance: mean={np.mean(f1_values):.4f}  std={np.std(f1_values):.4f}")

best_trainer   = seed_results[best_seed]["trainer"]
best_tokenizer = seed_results[best_seed]["tokenizer"]
best_model     = best_trainer.model

print("\n" + "="*70)
print("Diagnostic — per-seed validation errors (checking overlap)")
print("="*70)
val_texts_diag = list(val_df[text_col])
val_labels_diag = val_df["label_id"].values
for seed in SEEDS:
    m_diag = seed_results[seed]["trainer"].model
    tok_diag = seed_results[seed]["tokenizer"]
    m_diag.eval()
    logits_chunks = []
    with torch.no_grad():
        for i in range(0, len(val_texts_diag), 32):
            batch = val_texts_diag[i:i+32]
            enc = tok_diag(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
            logits_chunks.append(m_diag(**enc).logits.cpu().numpy())
    logits_diag = np.concatenate(logits_chunks, axis=0)
    preds_diag = logits_diag.argmax(axis=1)
    wrong_idx = np.where(preds_diag != val_labels_diag)[0]
    print(f"[seed={seed}] {len(wrong_idx)} wrong on val — row index(es): {list(val_df.index[wrong_idx])}")
print("If the same row index appears for all three seeds, the identical F1 reflects a genuine")
print("saturation effect (one hard/ambiguous example), not a training artifact — report it as such.")


## 7. Evaluation on AAAC-derived Test Set (prompt_injection-only task has no OSACT-side ground truth — OSACT never carries prompt_injection examples)

In [ ]:
def run_inference(df, tokenizer, model, text_column):
    model.eval()
    texts = list(df[text_column])
    all_logits, batch_size = [], 32
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
            all_logits.append(model(**enc).logits.cpu().numpy())
    logits = np.concatenate(all_logits, axis=0)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return probs.argmax(axis=1), probs

test_preds, test_probs = run_inference(test_df, best_tokenizer, best_model, text_col)
y_true = test_df["label_id"].values

print(classification_report(y_true, test_preds, target_names=["safe", "prompt_injection"], zero_division=0))
auc = roc_auc_score(y_true, test_probs[:, 1])
print(f"ROC-AUC: {auc:.4f}")

cm = confusion_matrix(y_true, test_preds)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", xticklabels=["safe","prompt_injection"], yticklabels=["safe","prompt_injection"], ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Adversarial Agent — AAAC test")
plt.tight_layout()
fig.savefig(f"{REPORT_DIR}/confusion_matrix_test.png", dpi=300)
plt.show()


## 8. Per-Language-Variant Robustness

In [ ]:
def wilson_ci(successes, n, alpha=0.05):
    if n == 0:
        return (float("nan"), float("nan"))
    return proportion_confint(successes, n, alpha=alpha, method="wilson")

lang_rows = []
for variant, sub in test_df.groupby("language_variant"):
    pos = test_df["language_variant"] == variant
    y_true_v = test_df.loc[pos, "label_id"].values
    y_pred_v = test_preds[pos.values]
    correct = (y_true_v == y_pred_v)
    lo, hi = wilson_ci(int(correct.sum()), len(correct))
    lang_rows.append({"language_variant": variant, "n": len(correct), "accuracy": correct.mean(),
                       "ci_low": lo, "ci_high": hi})

lang_df = pd.DataFrame(lang_rows)
lang_df.to_csv(f"{REPORT_DIR}/per_language_robustness.csv", index=False)
display(lang_df)


## 9. Threshold Calibration (replaces hard-coded PROMPT_INJECTION_CONF_THRESHOLD / PROMPT_INJECTION_MIN_MARGIN / OOD_ENTROPY_THRESHOLD — formerly named JAILBREAK_*)

In [ ]:
import sys
sys.path.insert(0, OUT_DIR)
from threshold_calibration import (calibrate_conf_margin_threshold, calibrate_ood_entropy_threshold,
                                    apply_manual_override_if_degenerate, export_threshold_json, batch_entropy)

# Manual fallback = the PRE-REBUILD production constants from adversarial_detector.py
# (formerly JAILBREAK_CONF_THRESHOLD=0.60, JAILBREAK_MIN_MARGIN=0.20, now renamed
# PROMPT_INJECTION_CONF_THRESHOLD / PROMPT_INJECTION_MIN_MARGIN) — used ONLY if the sweep
# itself reports DEGENERATE/UNRELIABLE, same discipline as the Intent Agent notebook's
# SENSITIVE/INJECTION override (never silently overrules a sweep that found real support).
MANUAL_FALLBACK = {"PROMPT_INJECTION_CONF_THRESHOLD": 0.60, "PROMPT_INJECTION_MIN_MARGIN": 0.20,
                   "OOD_ENTROPY_THRESHOLD": 0.60}  # pre-rebuild adversarial_detector.py constant

val_preds, val_probs = run_inference(val_df, best_tokenizer, best_model, text_col)
val_conf = val_probs.max(axis=1)
sorted_probs = np.sort(val_probs, axis=1)[:, ::-1]
val_margin = sorted_probs[:, 0] - sorted_probs[:, 1]
val_entropy = batch_entropy(val_probs)
val_is_correct = (val_preds == val_df["label_id"].values)

is_injection_pred = (val_preds == LABEL2ID["prompt_injection"])
y_true_injection = val_df["label_id"].values  # already binary: 1 = prompt_injection

conf_res, margin_res, sweep_df = calibrate_conf_margin_threshold(
    y_true_injection, is_injection_pred, val_conf, val_margin, threshold_name="PROMPT_INJECTION")

n_injection_val = int(y_true_injection.sum())
conf_res = apply_manual_override_if_degenerate(
    conf_res, MANUAL_FALLBACK["PROMPT_INJECTION_CONF_THRESHOLD"],
    f"only {n_injection_val} validation positives — no reliable interior optimum, same pattern as "
    f"the Intent Agent's SENSITIVE/INJECTION gates")
margin_res = apply_manual_override_if_degenerate(
    margin_res, MANUAL_FALLBACK["PROMPT_INJECTION_MIN_MARGIN"],
    f"same as PROMPT_INJECTION_CONF_THRESHOLD — see note above")

#  OOD entropy gate — combined val + dev_osact pool (mirrors the Intent Agent's fix) 
dev_osact_preds, dev_osact_probs = run_inference(dev_osact_adv, best_tokenizer, best_model, text_col)
dev_osact_entropy = batch_entropy(dev_osact_probs)
dev_osact_is_correct = (dev_osact_preds == dev_osact_adv["label_id"].values)  # true label is always "safe" here

combined_entropy = np.concatenate([val_entropy, dev_osact_entropy])
combined_is_correct = np.concatenate([val_is_correct, dev_osact_is_correct])
print(f"Entropy calibration pool: {len(val_entropy)} (val) + {len(dev_osact_entropy)} (dev_osact) "
      f"= {len(combined_entropy)} total, {int((~combined_is_correct).sum())} wrong "
      f"(was {int((~val_is_correct).sum())} wrong using val alone)")

ood_res, ood_sweep_df = calibrate_ood_entropy_threshold(combined_entropy, combined_is_correct, n_classes=NUM_LABELS)

n_wrong_combined = int((~combined_is_correct).sum())
ood_res = apply_manual_override_if_degenerate(
    ood_res, MANUAL_FALLBACK["OOD_ENTROPY_THRESHOLD"],
    f"only {n_wrong_combined} wrong prediction(s) in the combined {len(combined_entropy)}-row pool — "
    f"not enough errors to locate a real entropy boundary, same degeneracy pattern as the "
    f"PROMPT_INJECTION conf/margin gates")

all_results = {"PROMPT_INJECTION_CONF_THRESHOLD": conf_res, "PROMPT_INJECTION_MIN_MARGIN": margin_res,
               "OOD_ENTROPY_THRESHOLD": ood_res}
print("=" * 78)
for name, res in all_results.items():
    flag = "  ⚠ FLAGGED" if ("DEGENERATE" in res.notes or "UNRELIABLE" in res.notes) else (
           "  ✓ MANUAL" if "MANUAL_OVERRIDE" in res.notes else "  OK")
    print(f"{name:<28} = {res.chosen_value:<8}{flag}")
    if res.notes:
        print(f"    -> {res.notes}")
print("=" * 78)

export_threshold_json(
    f"{MODEL_OUT}/threshold.json",
    all_results,
    extra_meta={"calibrated_on": "layer1_validation.csv (PROMPT_INJECTION gates) + "
                                  "layer1_validation.csv+dev_osact.csv combined pool (OOD entropy gate)",
                "n_validation": int(len(val_df)), "n_dev_osact": int(len(dev_osact_adv)),
                "backbone": MODEL_CHECKPOINT, "num_labels": NUM_LABELS},
)
print(f"threshold.json written to {MODEL_OUT}/threshold.json")
print("MANUAL lines are stated judgment calls (sweep couldn't constrain them), not sweep results —")
print("cite them as such in the paper rather than presenting every number as empirically tuned.")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis_name, g in sweep_df.groupby("axis"):
    axes[0].plot(g["value"], g["f1"], label=axis_name)
axes[0].legend(); axes[0].set_title("prompt_injection conf/margin sweep")
axes[1].plot(ood_sweep_df["threshold"], ood_sweep_df["youden_j"])
axes[1].set_title("OOD entropy sweep (Youden's J) — combined val+dev_osact pool")
plt.tight_layout()
fig.savefig(f"{REPORT_DIR}/threshold_sweep_curves.png", dpi=300)
plt.show()


## 10. Save Model, Label Map, Metrics

In [ ]:
best_model.save_pretrained(MODEL_OUT)
best_tokenizer.save_pretrained(MODEL_OUT)

with open(f"{MODEL_OUT}/label_map.json", "w", encoding="utf-8") as f:
    json.dump({"id2label": {str(k): v for k, v in ID2LABEL.items()}, "label2id": LABEL2ID}, f, ensure_ascii=False, indent=2)

test_report = classification_report(y_true, test_preds, target_names=["safe","prompt_injection"], output_dict=True, zero_division=0)
metrics_summary = {
    "backbone": MODEL_CHECKPOINT,
    "num_labels": NUM_LABELS,
    "labels": list(ID2LABEL.values()),
    "label_derivation": "prompt_injection -> prompt_injection (unchanged); safe/hate_speech/sensitive_request -> safe; tool_misuse excluded",
    "best_seed": best_seed,
    "seed_val_macro_f1": {str(s): seed_results[s]["val_macro_f1"] for s in SEEDS},
    "seed_variance_std": float(np.std(f1_values)),
    "test_roc_auc": float(auc),
    "test_report": test_report,
    "per_language_robustness": lang_df.to_dict(orient="records"),
    "trained_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(f"{MODEL_OUT}/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_summary, f, ensure_ascii=False, indent=2, default=str)

print("Saved to:", MODEL_OUT)
print("Files:", os.listdir(MODEL_OUT))
